In [2]:
import pandas as pd
import torch
import os
file_path = r'data\rawData'
txt_file_path = r'data\processedData'
new_path = r'd:\Desktop\PHD\reasearch\biyework\maml'
os.chdir(new_path)
# 检查当前路径是否切换成功
current_path = os.getcwd()
print(current_path)

d:\Desktop\PHD\reasearch\biyework\maml


In [ ]:

# 读取CSV文件
csv_file_path = r'data\processedData\FLOOR3\all_data_new.csv'  # 替换为你的CSV文件路径
df = pd.read_csv(csv_file_path)
# 保存为PTH文件
pth_file_path = r"model\v1\input\FLOOR3_v3.pth"  
csv_file_path = r"model\v1\input\FLOOR3_v3.csv" #v3减掉了average_snr
# 读取 location_vector.csv 文件
location_vector_path = r'model\v1\output\location_vector_v2.csv'  # 替换为你的location_vector.csv文件路径
location_df = pd.read_csv(location_vector_path)

# 创建 location_id 到 idx 的映射
location_id_to_idx = dict(zip(location_df['location_id'], location_df['idx']))
print(df['location_id'].unique())
print(location_id_to_idx)
# 将 location_id 转换为 idx，如果 location_id 不在映射中，则丢弃
df['location_id'] = df['location_id'].map(location_id_to_idx)
# 丢弃 location_id 为 NaN 的行
df = df.dropna(subset=['location_id'])
# 打印转换后的 location_id 列，检查是否有丢失的点
print("Mapped Location IDs:")
print(len(df['location_id'].unique()))
print(df['location_id'].unique())
# 检查并转换DataFrame中的数据类型
df = df.apply(pd.to_numeric, errors='coerce').fillna(0).astype(float)
# 将sf全都除以12，将tp全都除以10
df['sf'] = df['sf'] / 13.0
df['tp'] = df['tp'] / 10.0
df['rssi'] = df['rssi'] / 150.0
df['average_rssi'] = df['average_rssi'] / 150.0
# 将snr和rssi_variance归一化
df['snr'] = (df['snr'] - df['snr'].min()) / (df['snr'].max() - df['snr'].min())
df['rssi_variance'] = (df['rssi_variance'] - df['rssi_variance'].min()) / (df['rssi_variance'].max() - df['rssi_variance'].min())

data_dict = {
    'rssi': torch.tensor(df[['rssi','average_rssi','rssi_variance',"snr",'rssi','average_rssi','rssi_variance',"snr"]].values, dtype=torch.float32),
    'snr': torch.tensor(df[["sf","tp"]].values, dtype=torch.float32),
    'sf': torch.tensor(df['sf'].values, dtype=torch.float32),
    'tp': torch.tensor(df['tp'].values, dtype=torch.float32),
    'label':  torch.tensor(df['location_id'].values, dtype=torch.int64)
}


# 确保数据长度是 batch_size * 2 * 16 的倍数
batch_size = 16  # 你可以根据需要调整 batch_size
total_length = len(df)
required_length = (total_length // (batch_size * 2 * 16)) * (batch_size * 2 * 16)

# 截断数据以匹配所需长度
data_dict['rssi'] = data_dict['rssi'][:required_length]
data_dict['snr'] = data_dict['snr'][:required_length]
data_dict['sf'] = data_dict['sf'][:required_length]
data_dict['tp'] = data_dict['tp'][:required_length]
data_dict['label'] = data_dict['label'][:required_length]
# pth的形式以字典的方式保存rssi,snr,sf,tp,location_id,label
torch.save(data_dict, pth_file_path)
# 保存同样的数据在csv文件中
df.to_csv(csv_file_path, index=False)

['1m' '302.0' '306-304' '308.0' '312' '322' '328' '330' '334' '336'
 '340.0' '344' '350' '354' '356.0' '360' '366.0' '370' 'point1' '308'
 '322.0' '340' '356' '302' '310' '316' '318' '320' '324' '326' '332' '338'
 '342' '346' '348' '350.0' '352.0' '358' '362' '364' '366' '368' '372'
 'point2' 'point3']
{'point1': 0, '370': 1, '366': 2, '360': 3, '356': 4, '354': 5, '350': 6, '344': 7, '340': 8, '336': 9, '334': 10, '1m': 11, '330': 12, '328': 13, '322': 14, '312': 15, '308': 16, '306-304': 17, '302': 18}
Mapped Location IDs:
19
[11. 17. 15. 14. 13. 12. 10.  9.  7.  6.  5.  3.  1.  0. 16.  8.  4. 18.
  2.]


In [6]:
# 验证一下这段代码
# 读取PTH文件
data_tensor = torch.load(pth_file_path, weights_only=True)
# 打印数据
print(data_tensor['rssi'])
print(data_tensor['rssi'].shape)
print(data_tensor['snr'].shape)
print(data_tensor['sf'].shape)
print(data_tensor['tp'].shape)
print(data_tensor['label'])
# 检查FLOOR3_v3.csv的label列存不存在12
print(sorted(df['location_id'].unique()))


tensor([[-0.3400, -0.3933,  0.2322,  0.9412],
        [-0.3933, -0.3933,  0.2322,  0.9412],
        [-0.3467, -0.3933,  0.2322,  0.9412],
        ...,
        [-0.7667, -0.8467,  0.0251,  0.3529],
        [-0.7800, -0.8467,  0.0251,  0.3529],
        [-0.7733, -0.8667,  0.0356,  0.2941]])
torch.Size([27648, 4])
torch.Size([27648, 2])
torch.Size([27648])
torch.Size([27648])
tensor([11, 11, 11,  ...,  1,  1,  1])
[np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(18.0)]
